In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, Dropout,
    Permute, Reshape, Bidirectional, GRU, TimeDistributed, Dense,
    LayerNormalization, Add, MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import ModelCheckpoint

from tensorflow.keras.layers import Input, Dense, Conv2D, BatchNormalization, Activation, MaxPooling2D, Dropout, Reshape, Permute, GlobalAveragePooling1D, Add, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.layers import Input, Dense, Conv2D, BatchNormalization, Activation, MaxPooling2D, Dropout, Permute, Reshape, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K



In [ ]:


from tensorflow.keras.layers import LayerNormalization, Dense, Dropout, MultiHeadAttention, Conv1D, Add, Multiply, Lambda, Activation
import tensorflow as tf

from tensorflow.keras.layers import MultiHeadAttention, Dense, LayerNormalization, Conv1D, Add, Dropout, Activation
from tensorflow.keras.layers import DepthwiseConv1D, BatchNormalization
def branchformer_block(x, head_size, num_heads, ff_dim, dropout=0.1, kernel_size=31, block_idx=0):
    prefix = f"branchformer{block_idx}"

    
    ff1 = Dense(ff_dim, activation='relu', name=f"{prefix}_ff1_dense1")(x)
    ff1 = Dropout(dropout, name=f"{prefix}_ff1_dropout")(ff1)
    ff1 = Dense(x.shape[-1], name=f"{prefix}_ff1_dense2")(ff1)
    x = Add(name=f"{prefix}_ff1_add")([x, Lambda(lambda z: 0.5 * z)(ff1)])

    
    x_ln_attn = LayerNormalization(epsilon=1e-6, name=f"{prefix}_attn_ln")(x)
    attn_out = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size,
        dropout=dropout,
        name=f"{prefix}_attn"
    )(x_ln_attn, x_ln_attn)
    attn_out = Dropout(dropout, name=f"{prefix}_attn_dropout")(attn_i4out)


     # ===== Local Branch: Conformer-style Conv Block =====


    # ===== Local Branch: Full Conformer-Style Conv Module =====
    conv_input = LayerNormalization(epsilon=1e-6, name=f"{prefix}_conv_ln")(x)

    # Pointwise Conv → GLU
    conv_pw = Conv1D(filters=2 * x.shape[-1], kernel_size=1, padding='same', name=f"{prefix}_conv_pw1")(conv_input)
    conv_u = Lambda(lambda z: z[:, :, :x.shape[-1]], name=f"{prefix}_glu_u")(conv_pw)
    conv_v = Lambda(lambda z: z[:, :, x.shape[-1]:], name=f"{prefix}_glu_v")(conv_pw)
    conv_v = Activation("sigmoid", name=f"{prefix}_glu_sigmoid")(conv_v)  # ✅ sigmoid 
    conv_glu = Multiply(name=f"{prefix}_glu_out")([conv_u, conv_v])  # GLU = u * v

    # Depthwise Conv
    conv_dw = DepthwiseConv1D(kernel_size=kernel_size, padding='same', name=f"{prefix}_depthwise")(conv_glu)
    conv_dw = BatchNormalization(name=f"{prefix}_dw_bn")(conv_dw)
    conv_dw = Activation('swish', name=f"{prefix}_swish")(conv_dw)
    # Pointwise Conv + Dropout
    conv_out = Conv1D(filters=x.shape[-1], kernel_size=1, padding='same', name=f"{prefix}_conv_pw2")(conv_dw)
    conv_out = Dropout(dropout, name=f"{prefix}_conv_dropout")(conv_out)
    
    # ===== Merge  =====
    merged = Add(name=f"{prefix}_merge")([attn_out, conv_out])
    x = Add(name=f"{prefix}_residual_merge")([x, merged])

    # ===== FFN  =====
    ff2 = Dense(ff_dim, activation='relu', name=f"{prefix}_ff2_dense1")(x)
    ff2 = Dropout(dropout, name=f"{prefix}_ff2_dropout")(ff2)
    ff2 = Dense(x.shape[-1], name=f"{prefix}_ff2_dense2")(ff2)
    x = Add(name=f"{prefix}_ff2_add")([x, Lambda(lambda z: 0.5 * z)(ff2)])

    
    x = LayerNormalization(epsilon=1e-6, name=f"{prefix}_ln_out")(x)
    return x



In [ ]:
from tensorflow.keras.layers import Input, Permute, Reshape, GlobalAveragePooling1D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_resnet_conformer_seld(input_shape=(61,256,6), num_layers=1, head_size=32, num_heads=4, ff_dim=256, dropout_rate=0.15, fnn_units=[128]):

    spec_input = Input(shape=input_shape)

    # 
    x = Conv2D(filters=64, kernel_size=(3, 3), padding='same')(spec_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(1, 8))(x)
    x = Dropout(dropout_rate)(x)

    x = Conv2D(filters=64, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(1, 8))(x)
    x = Dropout(dropout_rate)(x)

    x = Conv2D(filters=64, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(1, 2))(x)
    x = Dropout(dropout_rate)(x)

    # 
    x = Permute((1, 2, 3))(x)
    x = Reshape((input_shape[0], -1))(x)
    
    for i in range(num_layers):
        x = branchformer_block(x, head_size=head_size, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout_rate, block_idx=i)

    embedding_output = GlobalAveragePooling1D(name='embedding_output')(x)

    # SED Output (Binary)
    sed_output = Dense(fnn_units[0], activation='relu', name="sed_dense")(embedding_output)
    sed_output = Dropout(dropout_rate, name="sed_dropout")(sed_output)
    sed_output = Dense(1, activation='sigmoid', name='sed_output')(sed_output)

    # DOA Output (8-Class)
    doa_output = Dense(fnn_units[0], activation='relu', name="doa_dense")(embedding_output)
    doa_output = Dropout(dropout_rate, name="doa_dropout")(doa_output)
    doa_output = Dense(8, activation='softmax', name='doa_output')(doa_output)

    # 
    full_model = Model(inputs=spec_input, outputs=[sed_output, doa_output, embedding_output])
    train_model = Model(inputs=spec_input, outputs=[sed_output, doa_output])

    return full_model, train_model

# 
full_model, train_model = build_resnet_conformer_seld()

train_model.compile(
    optimizer=Adam(learning_rate=0.00008), 
    loss=['binary_crossentropy', masked_categorical_crossentropy],  
    loss_weights=[10.0, 10.0]  
)

train_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 61, 256,   │          0 │ -                 │
│ (InputLayer)        │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 61, 256,   │      3,520 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 61, 256,   │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 61, 256,   │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 61, 32,    │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 61, 32,    │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 61, 32,    │     36,928 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 61, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 61, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 61, 4, 64) │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 61, 4, 64) │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 61, 4, 64) │     36,928 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 61, 4, 64) │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 61, 4, 64) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 61, 2, 64) │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 61, 2, 64) │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 61, 2, 64) │          0 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 61, 128)   │          0 │ permute[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 365,129 (1.39 MB)

 Trainable params: 364,489 (1.39 MB)

 Non-trainable params: 640 (2.50 KB)

In [ ]:
# model_layer/branch/branchformer_branch.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple
from .base_branch import ExpertModel
from .transformer_branch import SensorPatches, PatchEncoder, ClassToken

class BranchformerBlock(nn.Module):
    """
    Branchformer 核心块：并行处理全局注意力分支和局部卷积分支。
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout_rate: float = 0.1, kernel_size: int = 31):
        super().__init__()
        
        # FFN 1 (Sandwich 结构)
        self.ff1 = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_ff, d_model)
        )
        
        # 全局分支：多头注意力
        self.attn_ln = nn.LayerNorm(d_model, eps=1e-6)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout_rate,
            batch_first=True
        )
        
        # 局部分支：Conformer 风格的卷积模块
        self.conv_ln = nn.LayerNorm(d_model, eps=1e-6)
        self.conv_pw1 = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)
        self.conv_dw = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, padding=kernel_size // 2, groups=d_model)
        self.dw_bn = nn.BatchNorm1d(d_model)
        self.conv_pw2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout_rate)
        
        # FFN 2
        self.ff2 = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_ff, d_model)
        )
        
        self.final_ln = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # FFN 1
        x = x + 0.5 * self.ff1(x)
        
        # 全局分支
        attn_in = self.attn_ln(x)
        attn_out, _ = self.attn(attn_in, attn_in, attn_in)
        
        # 局部分支 (B, T, C) -> (B, C, T)
        conv_in = self.conv_ln(x).transpose(1, 2)
        # GLU 激活
        conv_pw = self.conv_pw1(conv_in)
        u, v = torch.chunk(conv_pw, 2, dim=1)
        conv_glu = u * torch.sigmoid(v)
        # Depthwise Conv
        conv_dw = self.conv_dw(conv_glu)
        conv_dw = self.dw_bn(conv_dw)
        conv_dw = F.silu(conv_dw)
        # PW 2
        conv_out = self.conv_pw2(conv_dw).transpose(1, 2)
        
        # 合并分支
        x = x + attn_out + self.dropout(conv_out)
        
        # FFN 2
        x = x + 0.5 * self.ff2(x)
        return self.final_ln(x)

class BranchformerExpert(ExpertModel):
    """
    Branchformer 专家模块：参数与 TransformerExpert 对齐，支持 UCI 数据集。
    """
    def _build_model(self, **kwargs):
        # 参数对齐：支持从 config 中读取 d_model/nhead 等别名
        self.projection_dim = kwargs.get('projection_dim', kwargs.get('d_model', 192))
        self.num_heads = kwargs.get('num_heads', kwargs.get('nhead', 4))
        self.d_ff = kwargs.get('d_ff', kwargs.get('dim_feedforward', 768))
        self.num_layers = kwargs.get('num_layers', 4)
        self.dropout_rate = kwargs.get('dropout_rate', kwargs.get('dropout', 0.1))
        
        self.patch_size = kwargs.get('patch_size', 16)
        self.time_step = kwargs.get('time_step', 16)
        self.use_cls_token = kwargs.get('use_cls_token', True)
        self.kernel_size = kwargs.get('kernel_size', 31)

        # 1. 输入投影层 (处理 UCI 1D 特征)
        input_features = self.input_shape[1]
        self.input_projection = nn.Linear(input_features, self.projection_dim)
        
        # 2. 补丁提取与编码 (与 TransformerExpert 一致)
        self.patches = SensorPatches(self.projection_dim, self.patch_size, self.time_step)
        
        if self.use_cls_token:
            self.cls_token = ClassToken(self.projection_dim)
        
        time_steps = self.input_shape[0]
        n_patches = (time_steps - self.patch_size) // self.time_step + 1
        if self.use_cls_token: n_patches += 1
        self.patch_encoder = PatchEncoder(n_patches, self.projection_dim)
        
        # 3. Branchformer 层堆叠
        self.layers = nn.ModuleList([
            BranchformerBlock(self.projection_dim, self.num_heads, self.d_ff, self.dropout_rate, self.kernel_size)
            for _ in range(self.num_layers)
        ])
        
        self.layer_norm = nn.LayerNorm(self.projection_dim)
        
        # 4. 输出投影
        if self.projection_dim != self.output_dim:
            self.output_projection = nn.Linear(self.projection_dim, self.output_dim)
        else:
            self.output_projection = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, time_steps, features)
        x = self.input_projection(x)
        x = self.patches(x)
        
        if self.use_cls_token:
            x = self.cls_token(x)
            
        x = self.patch_encoder(x)
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.layer_norm(x)
        
        # 提取特征：CLS token 或平均池化
        features = x[:, 0] if self.use_cls_token else x.mean(dim=1)
        self.intermediate_features = features.detach()
        
        return self.output_projection(features)